In [21]:
import actionet.plotting
%load_ext autoreload
%autoreload 2

import actionet
import scipy
import numpy as np
import scanpy as sc
import anndata
import pandas as pd

import matplotlib.pyplot as plt
from IPython.display import display
import lets_plot
lets_plot.LetsPlot.setup_html(no_js=True)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
adata = anndata.read_h5ad("../data/test_adata_post.h5ad")
adata

AnnData object with n_obs × n_vars = 6790 × 14409
    obs: 'Barcode', 'CellLabel', 'assigned_archetype'
    var: 'ENSEMBL', 'Gene', 'Chromosome', 'Biotype'
    uns: 'action_params', 'log1p'
    obsm: 'C_merged', 'C_stacked', 'H_merged', 'H_stacked', 'action', 'action_B', 'archetype_footprint', 'colors_actionet', 'umap_2d_actionet', 'umap_3d_actionet'
    varm: 'action_A', 'action_V', 'archetype_feat_profile', 'archetype_feat_specificity_lower', 'archetype_feat_specificity_upper'
    layers: 'logcounts'
    obsp: 'actionet'

In [3]:
actionet.plot_umap(
    adata,
    color='CellLabel',
    color_source='obs',
    basis='umap_2d_actionet', legend=True)


In [22]:
X_out = actionet.impute_features(adata, features=["Drd2", "Drd1", "Mog", "Aqp4", "Chat", "Vcan", "Cats"], features_use="Gene", layer='logcounts', alpha=0.85)

Features missing: Cats


In [5]:
actionet.plot_umap(
    adata,
    color=X_out["Drd2"],
    color_source='obs', 
    basis='umap_2d_actionet', legend=True)

In [6]:
markers = actionet.find_markers(adata, adata.obs['CellLabel'], features_use="Gene", top_genes=30, return_type='dataframe')
annots_out = actionet.annotate_cells(adata, markers, layer = 'logcounts', method='vision', features_use='Gene', network_key='actionet', use_enrichment=True, use_lpa=False, n_threads=0)
adata.obs['annot'] = annots_out['labels']
adata.obs['annot_conf'] = annots_out['confidence']
adata.obsm['annot_enrichment'] = annots_out['enrichment']

Computing feature specificity ... done


In [7]:
actionet.plot_umap(
    adata,
    color='annot',
    color_source='obs',
    trans_attr=adata.obs['annot_conf'], 
    basis='umap_2d_actionet', legend=True)


In [8]:
actionet.plot_umap(
    adata,
    color='colors_actionet',
    color_source='obsm',
    trans_attr=adata.obs['annot_conf'], 
    basis='umap_2d_actionet', legend=True)

In [9]:
X = adata.X
if hasattr(X, "toarray"):
    X = X.toarray()
adata.obs["n_counts"] = np.asarray(X).sum(axis=1)
adata.obs["n_genes"] = (np.asarray(X) > 0).sum(axis=1)

actionet.plot_qc_violin(
    adata,
    keys=adata.obs["n_counts"],
    groupby="CellLabel",
    log_trans="log10",
    title="QC metrics",
)


In [23]:
actionet.plot_feature_expression(
    adata,
    features=["Drd2", "Drd1", "Cats", "Aqp4", "Chat", "Vcan"],
    features_use="Gene",
    alpha=0,
    layer="logcounts",
    basis="umap_2d_actionet",
    single_plot=True,
    n_threads=0, point_size=1
)
